In [8]:
import pandas as pd

train_path = "train.csv"
pred_path  = r"result\data\gold\run_id=run_20260226_001425\results_train.csv"

train = pd.read_csv(train_path)
preds = pd.read_csv(pred_path)

# Normalize labels -> y_true
label_map = {
    "consistent": 1,
    "contradict": 0,
    "contradiction": 0,   # in case you used this spelling anywhere
}
train["label_norm"] = (
    train["label"].astype(str).str.strip().str.lower()
)
train["y_true"] = train["label_norm"].map(label_map)

# Normalize predictions -> y_pred
preds["y_pred"] = pd.to_numeric(preds["prediction"], errors="coerce").astype("Int64")

# (Optional) de-dup if needed
train = train.drop_duplicates(subset=["id"])
preds = preds.drop_duplicates(subset=["id"])

# Merge on id
df = train.merge(preds[["id", "y_pred"]], on="id", how="inner")

# Basic sanity checks
missing_pred = set(train["id"]) - set(df["id"])
missing_label = df["y_true"].isna().sum()
if missing_pred:
    print(f"⚠️ Missing predictions for {len(missing_pred)} train IDs (showing up to 10): {sorted(list(missing_pred))[:10]}")
if missing_label:
    print(f"⚠️ {missing_label} rows have unmapped labels. Unique labels:", sorted(train["label_norm"].unique()))

# Drop rows where label or pred is missing
df_eval = df.dropna(subset=["y_true", "y_pred"]).copy()
df_eval["y_true"] = df_eval["y_true"].astype(int)
df_eval["y_pred"] = df_eval["y_pred"].astype(int)

# Accuracy
acc = (df_eval["y_true"] == df_eval["y_pred"]).mean()
print(f"Train accuracy: {acc:.4f}  ({(df_eval['y_true']==df_eval['y_pred']).sum()}/{len(df_eval)})")

# (Optional) confusion counts
tp = ((df_eval.y_true==1) & (df_eval.y_pred==1)).sum()
tn = ((df_eval.y_true==0) & (df_eval.y_pred==0)).sum()
fp = ((df_eval.y_true==0) & (df_eval.y_pred==1)).sum()
fn = ((df_eval.y_true==1) & (df_eval.y_pred==0)).sum()
print(f"TP={tp}, TN={tn}, FP={fp}, FN={fn}")


Train accuracy: 0.4500  (36/80)
TP=8, TN=28, FP=1, FN=43


In [9]:
from sklearn.metrics import classification_report
report = classification_report(df_eval["y_true"], df_eval["y_pred"], target_names=["contradict", "consistent"])

In [10]:
print(report)
print(f"TP={tp}, TN={tn}, FP={fp}, FN={fn}")

              precision    recall  f1-score   support

  contradict       0.39      0.97      0.56        29
  consistent       0.89      0.16      0.27        51

    accuracy                           0.45        80
   macro avg       0.64      0.56      0.41        80
weighted avg       0.71      0.45      0.37        80

TP=8, TN=28, FP=1, FN=43


In [7]:
import pandas as pd

df = pd.read_csv("result/data/gold/run_id=run_20260225_221138/decision_scores.csv")

print("Prediction distribution:\n", df["prediction"].value_counts(normalize=True))
print("\nStatus distribution:\n", df["status"].value_counts(normalize=True))
print("\nTop rationales:\n", df["rationale"].value_counts().head(10))

# “No evidence” bucket (often the #1 cause)
no_ev = (df["support_n"] + df["neutral_n"] == 0).mean()
print("\n% with no (support+neutral) evidence:", round(no_ev*100,2), "%")

# UNSAT bucket (often #2 cause)
unsat = (df["status"] == "UNSAT").mean()
print("% UNSAT:", round(unsat*100,2), "%")

Prediction distribution:
 prediction
1    0.892857
0    0.107143
Name: proportion, dtype: float64

Status distribution:
 status
SAT                0.928571
SOFT_VIOLATIONS    0.071429
Name: proportion, dtype: float64

Top rationales:
 rationale
Neutral/contradict mix; score penalizes contradictions.                                   108
Mixed evidence (support/neutral with contradictions); prediction uses score/threshold.     22
Only neutral evidence; treated as weak support (alpha=0.15).                                7
Supporting evidence present; no hard contradictions detected.                               2
Only contradicting evidence found; predicting 0.                                            1
Name: count, dtype: int64

% with no (support+neutral) evidence: 0.71 %
% UNSAT: 0.0 %
